# logit-thermo A2-10：凝固点降下（Pythia-1B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは約 2.5〜3 時間。終わったら自動で切れます。
- ゲート：`docs/A2_10_GATE.md`。腕は「なし」「なし（別の種）」「偽薬 7.5%・15%・30%」の 5 本、温度は 0.4〜1.05 の 14 個。モデルごとに 1 本のノートブック（3 枠で並べて回す）。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_10_collect.py`（`50570e317e20a34a…`）は埋め込み。
- 保存先 `A2_10/`。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。
- **通知**：🚀・腕ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-10 凝固点降下 データ集め（Pythia-1B、偽薬 7.5・15・30%、温度 0.4〜1.05、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-10 凝固点降下 データ集め（Pythia-1B、偽薬 7.5・15・30%、温度 0.4〜1.05、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-10 freezing point pythia', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0'})
SRC22, SRC24, SRC7C, SRC7BC = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC']
SRC10C = '"""A2-10（ゲート docs/A2_10_GATE.md）のデータ集め：凝固点降下（偽薬を濃くして、本当の融点 T_m と T_0 の両方の下がり方を測る）。GPU（L4）。\nscripts/a2_7b_collect.py を実行したあとの同じ名前空間で実行し、温度の並び・腕・種だけを置きかえる（generate_7b・run_7b・teacher_7b は\n実行するときに B7_TEMPS・B7_ARMS・SEED7B を読むので、禁止のしかたは A2-7b のまま）。ノートブックごとに C10_GROUP を決める。\n"""\nB7_TEMPS = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\nB7_ARMS = [(\'none\', \'none\', 0.0), (\'none_b\', \'none\', 0.0),\n           (\'pl075\', \'placebo\', 0.075), (\'pl150\', \'placebo\', 0.15), (\'pl300\', \'placebo\', 0.30)]\nSEED7B = 20261116\n\n\ndef self_test_10():\n    assert len(B7_TEMPS) == 14 and all(T in B7_TEMPS for T in [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05])\n    assert [a[0] for a in B7_ARMS] == [\'none\', \'none_b\', \'pl075\', \'pl150\', \'pl300\']\n    seeds = {gen_seed(k, s, a) for k in range(14) for s in range(2) for a in (\'none\', \'none_b\')}\n    assert len(seeds) == 56 and SEED7B == 20261116\n    return \'OK\'\n'
assert hashlib.sha256(SRC10C.encode()).hexdigest() == '50570e317e20a34a18d75c15fb18832906909b24252d0f82ce235eccffa9b386'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRC10C)
C10_GROUP = 'pythia'
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 5 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_10())
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_10', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = B7_GROUPS[C10_GROUP]
    run_7b(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_10', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(gate='docs/A2_10_GATE.md', group=C10_GROUP, temps=B7_TEMPS, arms=[a[0] for a in B7_ARMS], seed=SEED7B,
                   a2_7b_collect_sha256='b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', a2_10_collect_sha256='50570e317e20a34a18d75c15fb18832906909b24252d0f82ce235eccffa9b386', torch=torch.__version__,
                   transformers=transformers.__version__, python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0),
                   matmul=torch.get_float32_matmul_precision()), open('/content/drive/MyDrive/logit-thermo/A2_10' + f'/run_meta_{C10_GROUP}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-10 のデータ集め（Pythia-1B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-10 の Pythia-1B を A2_10 に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
